# Clasificación automática de tipos de preguntas mediante Deep Learning
**TFM — Trabajo 2.3**

Este notebook implementa el pipeline completo descrito en la propuesta del proyecto:

1. Carga y análisis exploratorio del dataset TREC Question Classification
2. Preprocesamiento de los datos
3. Modelo base: TF-IDF + Logistic Regression
4. Redes recurrentes: LSTM y BiLSTM (PyTorch)
5. Transformers: BERT y DistilBERT (fine-tuning con Hugging Face)
6. Comparación de resultados (Accuracy, Precision, Recall, F1-Score, matriz de confusión)

Diseñado para ejecutarse en Google Colab con GPU (Entorno de ejecución → Cambiar tipo de entorno → GPU).

Dataset: [TREC Question Classification Dataset (Kaggle)](https://www.kaggle.com/datasets/thedevastator/the-trec-question-classification-dataset-a-longi/data)


## 0. Instalación de dependencias
Ejecutar solo una vez (descomentar en Colab).

In [ ]:
# !pip install -q transformers datasets evaluate torch scikit-learn pandas numpy matplotlib seaborn


## 1. Imports y configuración

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

sns.set_theme(style="whitegrid")


## 2. Carga de los datos

Se asume que `train.csv` y `test.csv` están en el mismo directorio del notebook
(en Colab, súbelos con el panel de archivos o monta Google Drive).

In [ ]:
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
train_df.head()


In [ ]:
# Mapeo de las 6 categorías principales (label-coarse) del dataset TREC.
# Verificado inspeccionando ejemplos de cada clase y comparando con las
# estadísticas públicas del dataset TREC-6 (ABBR es la clase minoritaria, ~86 ejemplos).
LABEL_COARSE_MAP = {
    0: "DESC",  # Description / definición
    1: "ENTY",  # Entity / entidad
    2: "ABBR",  # Abbreviation / abreviatura
    3: "HUM",   # Human / persona
    4: "NUM",   # Numeric / número
    5: "LOC",   # Location / lugar
}

train_df["label_name"] = train_df["label-coarse"].map(LABEL_COARSE_MAP)
test_df["label_name"] = test_df["label-coarse"].map(LABEL_COARSE_MAP)

N_CLASSES = train_df["label-coarse"].nunique()
print("Número de clases:", N_CLASSES)
train_df["label_name"].value_counts()


## 3. Análisis exploratorio de los datos (EDA)

In [ ]:
# Comprobación de valores nulos y duplicados
print("Nulos en train:\n", train_df.isnull().sum())
print("\nDuplicados en train:", train_df.duplicated(subset=["text"]).sum())
print("Duplicados en test:", test_df.duplicated(subset=["text"]).sum())


In [ ]:
# Distribución de clases (label-coarse)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=train_df, x="label_name", order=LABEL_COARSE_MAP.values(), ax=axes[0])
axes[0].set_title("Distribución de clases - Train")
axes[0].set_xlabel("Categoría")

sns.countplot(data=test_df, x="label_name", order=LABEL_COARSE_MAP.values(), ax=axes[1])
axes[1].set_title("Distribución de clases - Test")
axes[1].set_xlabel("Categoría")
plt.tight_layout()
plt.show()


In [ ]:
# Longitud de las preguntas (en número de palabras)
train_df["n_words"] = train_df["text"].str.split().str.len()
print(train_df["n_words"].describe())

plt.figure(figsize=(8, 4))
sns.histplot(train_df["n_words"], bins=30, kde=True)
plt.title("Distribución de la longitud de las preguntas (en palabras)")
plt.xlabel("Número de palabras")
plt.show()


In [ ]:
# Número de subcategorías (label-fine)
print("Número de subcategorías (label-fine):", train_df["label-fine"].nunique())


## 4. Preprocesamiento

Se aplican las etapas descritas en la propuesta: limpieza de texto, conversión a
minúsculas, eliminación de caracteres irrelevantes y división en
entrenamiento / validación. El conjunto `test.csv` se reserva como conjunto de
prueba final, sin tocar hasta la evaluación final de cada modelo.

In [ ]:
def clean_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9À-ÿ\s'?.,]", " ", text)  # elimina caracteres especiales irrelevantes
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

train_df[["text", "clean_text"]].head()


In [ ]:
# División 85/15 de train -> train/validación (test.csv se usa como test final)
train_split, val_split = train_test_split(
    train_df, test_size=0.15, random_state=SEED, stratify=train_df["label-coarse"]
)
print("Train:", train_split.shape, "Val:", val_split.shape, "Test:", test_df.shape)


## 5. Modelo base: TF-IDF + Logistic Regression

Modelo tradicional de referencia (Machine Learning clásico), tal como se
describe en la sección 4.a de la propuesta.

In [ ]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(train_split["clean_text"])
X_val_tfidf = tfidf.transform(val_split["clean_text"])
X_test_tfidf = tfidf.transform(test_df["clean_text"])

y_train = train_split["label-coarse"].values
y_val = val_split["label-coarse"].values
y_test = test_df["label-coarse"].values

logreg = LogisticRegression(max_iter=1000, C=1.0)
logreg.fit(X_train_tfidf, y_train)


In [ ]:
def evaluate_predictions(y_true, y_pred, model_name, label_map=LABEL_COARSE_MAP):
    """Calcula y muestra las métricas estándar para un modelo."""
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    print(f"\n=== {model_name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision (macro): {precision:.4f}")
    print(f"Recall (macro)   : {recall:.4f}")
    print(f"F1-score (macro) : {f1:.4f}\n")
    print(classification_report(
        y_true, y_pred,
        target_names=[label_map[i] for i in sorted(label_map)],
        zero_division=0
    ))
    return {"model": model_name, "accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


def plot_confusion_matrix(y_true, y_pred, model_name, label_map=LABEL_COARSE_MAP):
    cm = confusion_matrix(y_true, y_pred)
    labels = [label_map[i] for i in sorted(label_map)]
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(f"Matriz de confusión - {model_name}")
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.tight_layout()
    plt.show()


results = []  # acumula los resultados de todos los modelos para la comparación final


In [ ]:
y_pred_logreg = logreg.predict(X_test_tfidf)
results.append(evaluate_predictions(y_test, y_pred_logreg, "Logistic Regression + TF-IDF"))
plot_confusion_matrix(y_test, y_pred_logreg, "Logistic Regression + TF-IDF")


## 6. Redes neuronales recurrentes: LSTM y BiLSTM

Implementación con PyTorch, tal como se describe en la sección 4.a de la
propuesta. Se construye un vocabulario propio a partir de los datos de
entrenamiento y se entrena un embedding desde cero.

In [ ]:
from collections import Counter

def tokenize(text):
    return text.split()

# Construcción del vocabulario a partir del conjunto de entrenamiento
counter = Counter()
for t in train_split["clean_text"]:
    counter.update(tokenize(t))

MIN_FREQ = 1
vocab = {"<pad>": 0, "<unk>": 1}
for word, freq in counter.items():
    if freq >= MIN_FREQ:
        vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)
print("Tamaño del vocabulario:", VOCAB_SIZE)

def encode(text, max_len=30):
    tokens = tokenize(text)[:max_len]
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokens]
    ids = ids + [vocab["<pad>"]] * (max_len - len(ids))
    return ids

MAX_LEN = 30


In [ ]:
class QuestionDataset(Dataset):
    def __init__(self, df, max_len=MAX_LEN):
        self.texts = df["clean_text"].values
        self.labels = df["label-coarse"].values
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode(self.texts[idx], self.max_len)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)


train_ds = QuestionDataset(train_split)
val_ds = QuestionDataset(val_split)
test_ds = QuestionDataset(test_df)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)


In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, n_classes=N_CLASSES,
                 bidirectional=False, n_layers=1, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=n_layers,
            bidirectional=bidirectional, batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0
        )
        mult = 2 if bidirectional else 1
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * mult, n_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded)
        if self.lstm.bidirectional:
            hidden_cat = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden_cat = hidden[-1]
        out = self.dropout(hidden_cat)
        return self.fc(out)


In [ ]:
def train_torch_model(model, train_loader, val_loader, n_epochs=10, lr=1e-3, patience=3):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(n_epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * x.size(0)
            train_correct += (logits.argmax(1) == y).sum().item()
            train_total += x.size(0)

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = model(x)
                loss = criterion(logits, y)
                val_loss += loss.item() * x.size(0)
                val_correct += (logits.argmax(1) == y).sum().item()
                val_total += x.size(0)

        train_loss /= train_total
        val_loss /= val_total
        train_acc = train_correct / train_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1}/{n_epochs} - "
              f"train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} - "
              f"val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping activado.")
                break

    model.load_state_dict(best_state)
    return model, history


def plot_learning_curves(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"], label="Val")
    axes[0].set_title(f"{model_name} - Loss")
    axes[0].set_xlabel("Época")
    axes[0].legend()

    axes[1].plot(history["train_acc"], label="Train")
    axes[1].plot(history["val_acc"], label="Val")
    axes[1].set_title(f"{model_name} - Accuracy")
    axes[1].set_xlabel("Época")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


@torch.no_grad()
def predict_torch_model(model, loader):
    model.eval()
    preds, trues = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        logits = model(x)
        preds.extend(logits.argmax(1).cpu().numpy())
        trues.extend(y.numpy())
    return np.array(trues), np.array(preds)


In [ ]:
# --- LSTM ---
lstm_model = LSTMClassifier(VOCAB_SIZE, bidirectional=False)
lstm_model, lstm_history = train_torch_model(lstm_model, train_loader, val_loader, n_epochs=15, lr=1e-3)
plot_learning_curves(lstm_history, "LSTM")

y_true_lstm, y_pred_lstm = predict_torch_model(lstm_model, test_loader)
results.append(evaluate_predictions(y_true_lstm, y_pred_lstm, "LSTM"))
plot_confusion_matrix(y_true_lstm, y_pred_lstm, "LSTM")


In [ ]:
# --- BiLSTM ---
bilstm_model = LSTMClassifier(VOCAB_SIZE, bidirectional=True)
bilstm_model, bilstm_history = train_torch_model(bilstm_model, train_loader, val_loader, n_epochs=15, lr=1e-3)
plot_learning_curves(bilstm_history, "BiLSTM")

y_true_bilstm, y_pred_bilstm = predict_torch_model(bilstm_model, test_loader)
results.append(evaluate_predictions(y_true_bilstm, y_pred_bilstm, "BiLSTM"))
plot_confusion_matrix(y_true_bilstm, y_pred_bilstm, "BiLSTM")


## 7. Modelos basados en Transformers: BERT y DistilBERT

Fine-tuning mediante transferencia de aprendizaje mediante la librería
Hugging Face `transformers`, tal como se describe en las secciones 4.b y 4.c
de la propuesta. Se recomienda ejecutar esta sección con GPU (Google Colab).

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import Dataset as HFDataset
import evaluate

# Datasets de Hugging Face a partir de los splits ya creados
hf_train = HFDataset.from_pandas(train_split[["clean_text", "label-coarse"]].rename(
    columns={"clean_text": "text", "label-coarse": "label"}
).reset_index(drop=True))
hf_val = HFDataset.from_pandas(val_split[["clean_text", "label-coarse"]].rename(
    columns={"clean_text": "text", "label-coarse": "label"}
).reset_index(drop=True))
hf_test = HFDataset.from_pandas(test_df[["clean_text", "label-coarse"]].rename(
    columns={"clean_text": "text", "label-coarse": "label"}
).reset_index(drop=True))

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1": f1}


In [ ]:
def fine_tune_transformer(model_checkpoint, output_dir, n_epochs=4, lr=2e-5, batch_size=16):
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    def tokenize_fn(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=32)

    train_tok = hf_train.map(tokenize_fn, batched=True)
    val_tok = hf_val.map(tokenize_fn, batched=True)
    test_tok = hf_test.map(tokenize_fn, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint, num_labels=N_CLASSES
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=n_epochs,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_steps=50,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    test_preds = trainer.predict(test_tok)
    y_pred = np.argmax(test_preds.predictions, axis=-1)
    y_true = np.array(test_tok["label"])

    return trainer, y_true, y_pred


In [ ]:
# --- DistilBERT ---
distilbert_trainer, y_true_distil, y_pred_distil = fine_tune_transformer(
    "distilbert-base-uncased", output_dir="./distilbert_output", n_epochs=4, lr=5e-5
)
results.append(evaluate_predictions(y_true_distil, y_pred_distil, "DistilBERT"))
plot_confusion_matrix(y_true_distil, y_pred_distil, "DistilBERT")


In [ ]:
# --- BERT ---
bert_trainer, y_true_bert, y_pred_bert = fine_tune_transformer(
    "bert-base-uncased", output_dir="./bert_output", n_epochs=4, lr=2e-5
)
results.append(evaluate_predictions(y_true_bert, y_pred_bert, "BERT"))
plot_confusion_matrix(y_true_bert, y_pred_bert, "BERT")


## 8. Comparación final de modelos

Tabla y gráfico comparativo de Accuracy, Precision, Recall y F1-Score
para todos los modelos evaluados (sección 6.c de la propuesta).

In [ ]:
results_df = pd.DataFrame(results).sort_values("f1", ascending=False).reset_index(drop=True)
results_df


In [ ]:
results_melted = results_df.melt(id_vars="model", var_name="metric", value_name="score")

plt.figure(figsize=(10, 5))
sns.barplot(data=results_melted, x="model", y="score", hue="metric")
plt.title("Comparación de modelos - TREC Question Classification")
plt.ylabel("Score")
plt.xlabel("Modelo")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 9. Criterio de validez para producción

Según la sección 6.e de la propuesta, un modelo se considera apto para
producción si:

- Accuracy > 90%
- F1-Score (macro) > 0.90
- Rendimiento estable entre validación y test (sin sobreajuste significativo)


In [ ]:
PROD_ACC_THRESHOLD = 0.90
PROD_F1_THRESHOLD = 0.90

results_df["apto_para_produccion"] = (
    (results_df["accuracy"] > PROD_ACC_THRESHOLD) &
    (results_df["f1"] > PROD_F1_THRESHOLD)
)
results_df


## 10. Guardado del mejor modelo

El mejor modelo (según F1-Score macro) se guarda para su posterior despliegue,
en formato compatible con Hugging Face (`.bin`/`.safetensors` + tokenizer) o
PyTorch (`.pt`), según corresponda (sección 8.b de la propuesta).

In [ ]:
best_model_name = results_df.iloc[0]["model"]
print("Mejor modelo:", best_model_name)

# Ejemplo de guardado si el mejor modelo es un Transformer:
# distilbert_trainer.save_model("./best_model_distilbert")
# tokenizer.save_pretrained("./best_model_distilbert")

# Ejemplo de guardado si el mejor modelo es LSTM/BiLSTM:
# torch.save(bilstm_model.state_dict(), "bilstm_model.pt")
